In [1]:
from functools import reduce
import re

import pandas as pd
import numpy as np

from src.categories import subcat_to_cat

import pickle

In [2]:
# preds_path_ls = [
#     ('raw 0 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_0sh.bin'),
#     ('raw 5 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_5sh.bin'),
#     ('LoRA 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg13_0sh.bin'),
#     ('LoRA 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg13_5sh.bin'),
#     ('GSOFT 0 shot', r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg0_0sh.bin'),
#     ('GSOFT 5 shot', r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg0_5sh.bin'),
# ]

preds_path_ls = [
    ('raw 0 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_0sh.bin'),
    ('raw 5 shot',   r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\Raw_5sh.bin'),
    ('LoRA 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg17_0sh.bin'),
    ('LoRA 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg17_5sh.bin'),
    ('LoRA-OOD 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-OOD-MMLU_0sh.bin'),
    ('LoRA-OOD 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\LoRA-cfg18-OOD-MMLU_5sh.bin'),
    # ('GSOFT-OOD 0 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-OOD-MMLU_0sh.bin'),
    ('GSOFT-OOD 5 shot',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\test_preds\GSOFT-cfg3-OOD-MMLU_5sh.bin'),
]

In [3]:
preds_dfs = {}

for run_name, preds_path in preds_path_ls:
    with open(preds_path, 'rb') as f:
        preds_dfs[run_name] = pickle.load(file=f)

        print(f"{run_name:16}: {preds_dfs[run_name].shape=}")

raw 0 shot      : preds_dfs[run_name].shape=(14042, 6)
raw 5 shot      : preds_dfs[run_name].shape=(14042, 6)
LoRA 0 shot     : preds_dfs[run_name].shape=(14042, 6)
LoRA 5 shot     : preds_dfs[run_name].shape=(14042, 6)
LoRA-OOD 0 shot : preds_dfs[run_name].shape=(14042, 7)
LoRA-OOD 5 shot : preds_dfs[run_name].shape=(14042, 7)
GSOFT-OOD 5 shot: preds_dfs[run_name].shape=(14042, 7)


In [4]:
idx = 23

print(preds_dfs['GSOFT-OOD 5 shot']['text'][idx])
print(preds_dfs['GSOFT-OOD 5 shot']['model_pred'][idx])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

The following are multiple choice questions (with answers) about abstract_algebra. Output 'A', 'B', 'C', or 'D'. Full answer not needed.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find all c in Z_3 such that Z_3[x]/(x^2 + c) is a field.
A. 0
B. 1
C. 2
D. 3<|eot_id|><|start_header_id|>assistant<|end_header_id|>

B<|eot_id|><|start_header_id|>user<|end_header_id|>

Statement 1 | If aH is an element of a factor group, then |aH| divides |a|. Statement 2 | If H and K are subgroups of G then HK is a subgroup of G.
A. True, True
B. False, False
C. True, False
D. False, True<|eot_id|><|start_header_id|>assistant<|end_header_id|>

B<|eot_id|><|start_header_id|>user<|end_header_id|>

Statement 1 | Every element of a group generates a cyclic subgroup of the group. Statement 2 | The symmetric group S_10 has 10 elements.
A. True, True
B. False, False
C. True, False
D. False, True<|eot_id|><|start_header_id|>assistant<|end_header_i

In [5]:
preds_dfs['GSOFT-OOD 5 shot']['model_pred'].apply(
    lambda p: p['generated_text'][0]
).value_counts()

model_pred
B    3974
C    3664
A    3439
D    2963
T       2
Name: count, dtype: int64

In [6]:
def get_category(subject):
    return subcat_to_cat[subject]

OOD_part = re.compile('The correct answer is [ABCD]')
OOD_part_len = len('The correct answer is ')

def get_pred(p):
    if OOD_part.search(p):
        return p[OOD_part_len]
    else:
        return p[0]

def processor(row: pd.DataFrame):
    generated_text = row['model_pred']['generated_text']
    generated_text = generated_text.strip()
    

    # generated_text	subject	pred	true	corr	category
    category = get_category(row['subject'])
    pred = get_pred(generated_text)
    true = chr(ord('A') + row['answer'])
    corr = int(pred == true)

    return {
        'generated_text': generated_text,
        'subject': row['subject'],
        'pred': pred,
        'true': true,
        'corr': corr,
        'category': category
    }

In [7]:
preds_dfs['LoRA-OOD 0 shot'] = preds_dfs['LoRA-OOD 0 shot'].apply(processor, axis=1, result_type='expand')
preds_dfs['LoRA-OOD 5 shot'] = preds_dfs['LoRA-OOD 5 shot'].apply(processor, axis=1, result_type='expand')
# preds_dfs['GSOFT-OOD 0 shot'] = preds_dfs['GSOFT-OOD 0 shot'].apply(processor, axis=1, result_type='expand')
preds_dfs['GSOFT-OOD 5 shot'] = preds_dfs['GSOFT-OOD 5 shot'].apply(processor, axis=1, result_type='expand')

In [8]:
preds_dfs['LoRA-OOD 0 shot'].head()

,generated_text,subject,pred,true,corr,category
0,The correct answer is D. 6. The degree of the ...,abstract_algebra,D,B,0,STEM
1,The correct answer is B. 2. The answer is B be...,abstract_algebra,B,C,0,STEM
2,"The correct answer is D. 0,4. The polynomial x^5",abstract_algebra,D,D,1,STEM
3,"The correct answer is A. True, True.\nStatemen...",abstract_algebra,A,B,0,STEM
4,The correct answer is B. 6x^2 + 4x +,abstract_algebra,B,B,1,STEM


In [9]:
preds_dfs['LoRA-OOD 5 shot'].head()

,generated_text,subject,pred,true,corr,category
0,B,abstract_algebra,B,B,1,STEM
1,C\nSolution: The order of p is 4. The order of S,abstract_algebra,C,C,1,STEM
2,D\nAnswer format: A/B/C/D. Full answer not nee...,abstract_algebra,D,D,1,STEM
3,D,abstract_algebra,D,B,0,STEM
4,B\nSolution: f(x) = 4x - 5 =,abstract_algebra,B,B,1,STEM


In [10]:
preds_dfs['GSOFT-OOD 5 shot'].head()

,generated_text,subject,pred,true,corr,category
0,B,abstract_algebra,B,B,1,STEM
1,D,abstract_algebra,D,C,0,STEM
2,D,abstract_algebra,D,D,1,STEM
3,D,abstract_algebra,D,B,0,STEM
4,B. 6,abstract_algebra,B,B,1,STEM


In [11]:
acc_by_subjects = pd.DataFrame({
    'subject': list(set.union(
        *map(
            lambda df: set(df['subject']),
            preds_dfs.values()
        )
    ))
})

print(f"Subjects: {acc_by_subjects['subject']}")

for run_name, preds_df in preds_dfs.items():
    acc_df = preds_df[['subject', 'corr']].groupby(['subject'], as_index=False).mean()
    acc_df.rename(columns={'corr': run_name}, inplace=True)

    acc_by_subjects = acc_by_subjects.merge(
        acc_df,
        left_on='subject',
        right_on='subject',
        how='right'
    )

Subjects: 0                             human_aging
1                 professional_accounting
2              high_school_macroeconomics
3                         college_biology
4                         business_ethics
5     high_school_government_and_politics
6                         college_physics
7                       us_foreign_policy
8                  elementary_mathematics
9                       college_chemistry
10             high_school_microeconomics
11                             management
12                        world_religions
13                       medical_genetics
14                professional_psychology
15                 high_school_us_history
16                       machine_learning
17                 electrical_engineering
18                      computer_security
19                     clinical_knowledge
20                              marketing
21                     conceptual_physics
22                    high_school_biology
23                      

In [12]:
acc_by_subjects

,subject,raw 0 shot,raw 5 shot,LoRA 0 shot,LoRA 5 shot,LoRA-OOD 0 shot,LoRA-OOD 5 shot,GSOFT-OOD 5 shot
0,abstract_algebra,0.350000,0.310000,0.310000,0.340000,0.260000,0.300000,0.300000
1,anatomy,0.674074,0.666667,0.629630,0.614815,0.607407,0.637037,0.681481
2,astronomy,0.750000,0.723684,0.717105,0.730263,0.723684,0.677632,0.756579
3,business_ethics,0.660000,0.670000,0.610000,0.650000,0.730000,0.640000,0.740000
4,clinical_knowledge,0.716981,0.747170,0.743396,0.769811,0.698113,0.698113,0.724528
5,college_biology,0.715278,0.756944,0.784722,0.791667,0.715278,0.729167,0.784722
6,college_chemistry,0.450000,0.460000,0.480000,0.490000,0.410000,0.400000,0.440000
7,college_computer_science,0.470000,0.570000,0.470000,0.540000,0.490000,0.470000,0.490000
8,college_mathematics,0.350000,0.350000,0.280000,0.380000,0.260000,0.270000,0.350000
9,college_medicine,0.618497,0.641618,0.664740,0.647399,0.560694,0.566474,0.606936


In [13]:
acc_by_categories = pd.DataFrame({
    'category': list(set.union(
        *map(
            lambda df: set(df['category']),
            preds_dfs.values()
        )
    ))
})

for run_name, preds_df in preds_dfs.items():
    acc_df = preds_df[['category', 'corr']].groupby(['category'], as_index=False).mean()
    acc_df.rename(columns={'corr': run_name}, inplace=True)

    acc_by_categories = acc_by_categories.merge(
        acc_df,
        left_on='category',
        right_on='category',
        how='right'
    )

In [14]:
acc_by_categories.loc[:, list(filter(
    lambda s: ('5 shot' not in s),
    acc_by_categories.columns
))]

,category,raw 0 shot,LoRA 0 shot,LoRA-OOD 0 shot
0,STEM,0.525182,0.540755,0.505302
1,humanities,0.527099,0.556217,0.533688
2,"other (business, health, misc.)",0.706663,0.709130,0.690623
3,social sciences,0.721807,0.746831,0.721807


In [15]:
acc_by_categories.loc[:, list(filter(
    lambda s: ('0 shot' not in s),
    acc_by_categories.columns
))]

,category,raw 5 shot,LoRA 5 shot,LoRA-OOD 5 shot,GSOFT-OOD 5 shot
0,STEM,0.546720,0.561962,0.514248,0.539430
1,humanities,0.603826,0.591711,0.543677,0.586823
2,"other (business, health, misc.)",0.724244,0.719926,0.684454,0.713757
3,social sciences,0.758531,0.766006,0.714007,0.744556


In [16]:
total_res = pd.DataFrame(columns=['run_name', 'accuracy', 'correctness'])

for run_name, preds_df in preds_dfs.items():
    total_accuracy = preds_df['corr'].mean()
    total_correctness = (preds_df['pred'] != 'I').mean()

    total_res = pd.concat([
        total_res,
        pd.DataFrame({
            'run_name': [run_name],
            'accuracy': [total_accuracy],
            'correctness': [total_correctness],
        })
    ])

C:\Users\Vladimir\AppData\Local\Temp\ipykernel_27408\2066953499.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  total_res = pd.concat([


In [17]:
total_res

,run_name,accuracy,correctness
0,raw 0 shot,0.610810,0.997151
0,raw 5 shot,0.653255,0.999217
0,LoRA 0 shot,0.629967,0.992024
0,LoRA 5 shot,0.653112,1.000000
0,LoRA-OOD 0 shot,0.605042,1.000000
0,LoRA-OOD 5 shot,0.607178,1.000000
0,GSOFT-OOD 5 shot,0.640507,1.000000
